# Préprocessing du dataset


In [ ]:
from trafficsigns.config import ROOT
from trafficsigns.data.preprocessing import build_dataset, save_dataset
from pathlib import Path

raw_dir = ROOT / "data" / "raw" / "gtsrb" / "Train"
output_dir = ROOT / "data" / "processed"

# Dataset classique
img_X, img_y, img_paths = build_dataset(raw_dir)
save_dataset(img_X, img_y, img_paths, Path.joinpath(output_dir, "train"))


In [ ]:
from trafficsigns.config import ROOT
from trafficsigns.data.data_augmentation import balance_and_augment_dataset
from trafficsigns.data.preprocessing import build_dataset, save_dataset
from pathlib import Path

raw_dir = ROOT / "data" / "raw" / "gtsrb" / "Train"
output_dir = ROOT / "data" / "processed"

# Creation par data augmentation
# 1. Chargement/Build brut
X, y, paths = build_dataset(raw_dir)
print(f"Dataset brut : {X.shape[0]} images au total.")

# 2. Augmentation & Rééquilibrage (ex: 2000 images par classe = 86 000 images)
X_aug, y_aug, paths_aug = balance_and_augment_dataset(X, y, paths, target_per_class=2000)

# 3. Sauvegarde dans le dossier processed/train sous forme .npy
save_dataset(X_aug, y_aug, paths_aug, Path.joinpath(output_dir, "train_augmented"))
print(f"Dataset augmenté sauvegardé sous '{output_dir}'.")
print(f"x_augmented shape: {X_aug.shape}")
print(f"y_augmented shape: {y_aug.shape}")

In [ ]:
# split des valeurs

from trafficsigns.config import load_config
from trafficsigns.data.dataset import load_dataset
from trafficsigns.data.split import grouped_stratified_split, assert_no_leakage, save_split, val_fraction_per_class, extract_group_keys
from trafficsigns.config import ROOT
import numpy as np

cfg_path = Path.joinpath(ROOT, "configs", "exp_configs", "db_aug.yaml")
cfg = load_config(cfg_path) if cfg_path.exists() else {}
loaded_seed = cfg["seed"]
loaded_val_ratio = cfg["split"]["val_ratio"]

processed = Path.joinpath(ROOT, cfg["data"]["processed_dir"])
_, new_y, paths = load_dataset(processed, mmap=True)
new_groups = extract_group_keys(paths)

new_train_idx, new_val_idx = grouped_stratified_split(new_y, new_groups, loaded_val_ratio, loaded_seed)
assert_no_leakage(new_groups, new_train_idx, new_val_idx)
save_split(new_train_idx, new_val_idx, processed, loaded_seed, loaded_val_ratio)

print(f"train : {len(new_train_idx)} images, {len(np.unique(new_groups[new_train_idx]))} panneaux")
print(f"val   : {len(new_val_idx)} images, {len(np.unique(new_groups[new_val_idx]))} panneaux "
          f"({len(new_val_idx) / len(new_y):.1%})")
frac = val_fraction_per_class(new_y, new_val_idx)
print(f"part de validation par classe : min {frac.min():.1%}, max {frac.max():.1%}")

In [ ]:
# Création du modèle selon la configuration du fichier
from trafficsigns.config import ROOT, load_config
from trafficsigns.training.train import train_model

cfg_path = Path.joinpath(ROOT, "configs", "exp_configs", "db_aug.yaml")
cfg = load_config(cfg_path) if cfg_path.exists() else {}

# Extraire les sous-sections du YAML avec valeurs de secours (fallbacks)
data_cfg = cfg.get("data", {})
model_cfg = cfg.get("model", {})

print(data_cfg.get("processed_dir", "data/processed/train"))
train_model(
    model_to_train=model_cfg.get("type", "baseline"),
    data_dir=Path.joinpath(ROOT, data_cfg.get("processed_dir", "data/processed/train")),
    model_save_path=Path.joinpath(ROOT, model_cfg.get("dir", "models/new_model")),
    epochs=data_cfg.get("epochs", 15),
    batch_size=data_cfg.get("batch_size", 32)
)

In [3]:
from trafficsigns.config import ROOT, load_config
from trafficsigns.evaluation.metrics import evaluate_model
from pathlib import Path

cfg = load_config(Path(ROOT) / "configs" / "base.yaml")

raw_dir = ROOT / cfg["data"]["raw_dir"]
evaluate_model(
    model_path=ROOT / cfg["model"]["dir"] / "model.keras",
    test_csv=raw_dir / cfg["data"]["test_csv"],
    img_dir=raw_dir,
    train_dir=ROOT / cfg["data"]["processed_dir"],
    out_dir=ROOT / cfg["reports"]["result_dir"] / cfg["model"]["name"],
)

--- Modèle : S:\IDEProjects\CNN_roadsigns\models\cnn\model.keras
Tenseur de test : (12630, 48, 48, 3)
395/395 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step

Accuracy globale : 96.52%

              precision    recall  f1-score   support

           0       1.00      0.98      0.99        60
           1       0.96      0.99      0.98       720
           2       0.95      1.00      0.97       750
           3       0.93      0.99      0.96       450
           4       0.98      0.98      0.98       660
           5       0.96      0.92      0.94       630
           6       0.99      0.89      0.94       150
           7       1.00      0.88      0.94       450
           8       0.89      0.96      0.93       450
           9       0.96      0.99      0.97       480
          10       1.00      0.98      0.99       660
          11       0.97      1.00      0.98       420
          12       0.99      0.98      0.99       690
          13       0.98      1.00      0.99       720
          14      